<a href="https://colab.research.google.com/github/BrianCarela/Data-analytics-practice/blob/main/02_pandas_and_numpy/02_00_dataframes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐼 02_00 — DataFrames: A Complete Walkthrough
### Data Analyst Curriculum | Pandas Foundation

---

### How to use this notebook
- **Read every explanation** before running the code.
- **Run every cell top to bottom** — later cells depend on earlier ones.
- **Do every exercise** marked 🏋️ before moving on.
- **Keep the data small on purpose.** You can see every row, so nothing is magic.

### What you'll learn
1. What a DataFrame is and how to create one
2. Inspecting a DataFrame
3. Selecting columns and rows
4. Filtering rows (boolean indexing)
5. Adding new columns with `.apply()`
6. Summarizing with `.groupby()`
7. Sorting and finding extremes (`.sort_values()`, `.idxmax()`, `.idxmin()`)
8. Chaining operations together
9. A mini capstone exercise

---

## 🔧 Section 0: Setup

Just pandas for this notebook. Run this first.

In [ ]:
import pandas as pd

print('pandas loaded:', pd.__version__)

## 📋 Section 1: What Is a DataFrame?

A **DataFrame** is a table — rows and columns, exactly like a spreadsheet.

- Each **column** has a name and holds one type of data (numbers, text, True/False)
- Each **row** is one record — one person, one sale, one event

You'll almost always create DataFrames by loading a file (CSV, Excel), but the
clearest way to understand them is to build one from scratch using a plain
Python dictionary.

**The pattern:**
```
{ 'column_name': [list of values], ... }
```
Each key becomes a column name. Each list becomes the column's values.
The lists must all be the same length — one value per row.

In [ ]:
# A small café dataset — 8 orders, easy to see everything at once
data = {
    'item':     ['Latte', 'Espresso', 'Muffin', 'Latte', 'Tea', 'Muffin', 'Espresso', 'Tea'],
    'category': ['drink', 'drink', 'food', 'drink', 'drink', 'food', 'drink', 'drink'],
    'price':    [4.50, 3.00, 2.75, 4.50, 2.50, 2.75, 3.00, 2.50],
    'quantity': [2, 1, 3, 1, 2, 1, 2, 4],
}

df = pd.DataFrame(data)
df

Notice pandas automatically adds a **row index** on the left (0, 1, 2...).
That's just the default row label — you didn't put it there.

## 🔍 Section 2: Inspecting a DataFrame

These are the first things you run on *any* dataset, big or small.
Build the habit now.

In [ ]:
# Shape: (rows, columns)
print('Shape:', df.shape)
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])

In [ ]:
# Column names and data types
df.info()

In [ ]:
# Summary stats for numeric columns
df.describe()

In [ ]:
# First N rows (default 5)
df.head(3)

In [ ]:
# Last N rows
df.tail(3)

In [ ]:
# Check for missing values — always do this on real data
df.isnull().sum()

## 🎯 Section 3: Selecting Columns and Rows

### 3a. Selecting Columns

Two syntaxes:
- `df['column']` → one column → returns a **Series** (a labeled list)
- `df[['col1', 'col2']]` → multiple columns → returns a **DataFrame**

The double brackets on multi-column selects trip everyone up at first.
The outer `[]` is the selection operator. The inner `[]` is just a Python list.

In [ ]:
# Single column → Series
df['item']

In [ ]:
# Multiple columns → DataFrame (note the double brackets)
df[['item', 'price']]

In [ ]:
# A Series has math methods built in
print('Average price: $', df['price'].mean())
print('Total quantity sold:', df['quantity'].sum())
print('Most expensive item: $', df['price'].max())

### 3b. Selecting Rows with `.iloc[]` and `.loc[]`

- `.iloc[n]` — select by **position** (integer index, like a list)
- `.loc[n]` — select by **label** (the index value shown on the left)

With the default numeric index, they look the same — but they're different concepts.

In [ ]:
# .iloc — by position (0-based, like a list)
df.iloc[0]      # first row

In [ ]:
# .iloc with a range
df.iloc[0:3]    # rows 0, 1, 2

In [ ]:
# .loc — by label
df.loc[0]       # row with index label 0 (same here, but not always)

## 🔎 Section 4: Filtering Rows (Boolean Indexing)

This is the pandas equivalent of SQL's `WHERE` clause.

**How it works — two steps:**
1. A condition on a column produces a **True/False Series**
2. Passing that into `df[...]` keeps only the `True` rows

In [ ]:
# Step 1: the condition alone — see what it produces
df['category'] == 'drink'

In [ ]:
# Step 2: use it to filter
drinks = df[df['category'] == 'drink']
drinks

In [ ]:
# Filter for items priced over $3
pricey = df[df['price'] > 3.00]
pricey

### Multiple Conditions

Use `&` for AND, `|` for OR. **Wrap each condition in `()`** — required syntax.

In [ ]:
# Drinks that cost more than $3
pricey_drinks = df[(df['category'] == 'drink') & (df['price'] > 3.00)]
pricey_drinks

In [ ]:
# Food OR items costing less than $3
cheap_or_food = df[(df['category'] == 'food') | (df['price'] < 3.00)]
cheap_or_food

### 🏋️ Exercise 1

Filter `df` to only rows where `quantity` is greater than 1.
Then print how many rows matched and the average price of those items.

```python
# Your code here
```

In [ ]:
# Your code here

## ➕ Section 5: Adding New Columns with `.apply()`

`.apply()` takes a function and runs it on every value in a column,
returning a new Series. You then assign that to a new column name.

**The pattern:**
```python
df['new_column'] = df['existing_column'].apply(some_function)
```

You pass the *function name* — no parentheses, no arguments.
Pandas does the calling, row by row.

In [ ]:
# A function that labels a price as budget, mid, or premium
def price_tier(price):
    if price < 3.00:
        return 'budget'
    elif price <= 4.00:
        return 'mid'
    else:
        return 'premium'

# Apply it to every value in the 'price' column
df['tier'] = df['price'].apply(price_tier)
df

In [ ]:
# You can apply to any column — here, label quantity as single or bulk
def order_size(qty):
    if qty == 1:
        return 'single'
    else:
        return 'bulk'

df['order_size'] = df['quantity'].apply(order_size)
df

Notice the DataFrame now has two new columns — `tier` and `order_size`.
You built them from existing data using plain Python functions.
This is called **feature engineering** — creating new information from what you have.

### 🏋️ Exercise 2

Write a function called `is_expensive` that returns `True` if a price is
above $3.50, and `False` otherwise.

Apply it to the `price` column and store the result in a new column called `expensive`.

Then print how many items are marked expensive using `.sum()`
*(True counts as 1, False as 0 — so summing a boolean column counts the Trues).*

```python
# Your code here
```

In [ ]:
# Your code here

## 📊 Section 6: Summarizing with `.groupby()`

`.groupby()` splits the DataFrame into groups based on a column's values,
then lets you run an aggregation on each group.

This is SQL's `GROUP BY`. It's the most important pandas operation for analysts.

**The pattern:**
```python
df.groupby('group_column')['value_column'].aggregation()
```

Common aggregations: `.mean()`, `.sum()`, `.count()`, `.min()`, `.max()`, `.median()`

In [ ]:
# Average price per category
df.groupby('category')['price'].mean()

In [ ]:
# Total quantity sold per item
df.groupby('item')['quantity'].sum()

In [ ]:
# Multiple stats at once using .agg()
df.groupby('category')['price'].agg(['mean', 'min', 'max', 'count'])

In [ ]:
# Group by TWO columns
# .unstack() pivots the inner group into columns — easier to read
df.groupby(['category', 'tier'])['quantity'].sum().unstack()

### 🏋️ Exercise 3

1. Find the total revenue per item. Revenue = price × quantity.
   First add a `revenue` column to `df`, then use `.groupby()`.
2. Which item generated the most total revenue?

```python
# Your code here
```

In [ ]:
# Your code here

## 🔢 Section 7: Sorting and Finding Extremes

### 7a. Sorting

In [ ]:
# Sort by price, lowest to highest
df.sort_values('price')

In [ ]:
# Sort by price, highest to lowest
df.sort_values('price', ascending=False)

In [ ]:
# Sort by multiple columns — category first, then price within each category
df.sort_values(['category', 'price'], ascending=[True, False])

### 7b. `.idxmax()` and `.idxmin()`

These return the **index label** of the row with the highest or lowest value.
That label can then be passed to `.loc[]` to retrieve the full row.

In [ ]:
# Which row has the highest price?
idx = df['price'].idxmax()
print('Row index of highest price:', idx)
print()
print(df.loc[idx])

In [ ]:
# Which row has the lowest quantity?
idx = df['quantity'].idxmin()
print('Row index of lowest quantity:', idx)
print()
print(df.loc[idx])

### `.idxmax()` on a GroupBy result

After a `.groupby()`, the result is itself a Series — so you can call `.idxmax()`
on that too. This is how you answer questions like *"which group has the highest average?"*

In [ ]:
# Average price per category
avg_by_cat = df.groupby('category')['price'].mean()
print(avg_by_cat)
print()

# Which category has the higher average price?
top_category = avg_by_cat.idxmax()
print(f'The pricier category is: {top_category}')
print(f'Average price: ${avg_by_cat[top_category]:.2f}')

### 🏋️ Exercise 4

Using the `revenue` column you made in Exercise 3:
1. Find the index of the row with the highest revenue using `.idxmax()`
2. Use `.loc[]` to print that full row
3. Print a sentence: *"The best-selling order was X, generating $Y in revenue."*

```python
# Your code here
```

In [ ]:
# Your code here

## ⛓️ Section 8: Chaining Operations

Pandas is designed so you can chain methods — call one after another on the same line.
Each method returns a result, and the next method runs on that result.

This keeps your code concise and readable once you're comfortable.

In [ ]:
# Filter → groupby → aggregate → sort — all in one chain
(df[df['category'] == 'drink']
   .groupby('item')['quantity']
   .sum()
   .sort_values(ascending=False))

In [ ]:
# Most common tier among bulk orders
(df[df['order_size'] == 'bulk']
   .groupby('tier')['quantity']
   .sum()
   .sort_values(ascending=False))

The parentheses around the whole expression let you break it across lines
for readability — Python treats it as one continuous expression.

## 🏁 Section 9: Mini Capstone

Put it all together. Answer these questions about the café dataset using pandas:

1. Add a `revenue` column if you haven't already (`price × quantity`)
2. What is the average revenue per `tier`? Which tier earns the most on average?
3. Among `bulk` orders only, which `item` has the highest total revenue?
4. Write 2–3 print statements summarizing your findings as if presenting to the café owner.

*Same mindset as 02_01's milestone: find a number → give it meaning → communicate it.*

```python
# Your code here
```

In [ ]:
# Your code here

---
## ✅ You're ready for 02_01

You've now seen every core DataFrame operation in a controlled environment:
- Building and inspecting DataFrames
- Selecting columns and rows
- Filtering with boolean indexing
- Adding columns with `.apply()`
- Summarizing with `.groupby()`
- Sorting and finding extremes with `.idxmax()` / `.idxmin()`
- Chaining operations

Go back to **02_01** and Exercise 2 should feel straightforward now.
The insurance dataset is just more rows — the operations are identical.